# Notebook 04 — Contamination probe

Checks whether the four judge models have memorised the MIRACL queries. Each model sees the first half of every sampled query and is asked to complete it exactly; we measure how often it reproduces the rest word for word. Low rates mean memorisation is unlikely to explain the results.

**Kaggle setup:** GPU T4 x2, Internet On, `HF_TOKEN` secret attached (Llama, Gemma and Aya are gated).
**Inputs:** attach the **same inputs as notebook 03** (the six `pairs_*.jsonl` files).
**Runs all four models in one version** (about 30–60 minutes). Output: `contamination_probe.csv` and `contamination_summary.csv`.

## 0. Install packages

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas<3", "scikit-learn<1.9",
                "-U", "transformers", "accelerate", "bitsandbytes", "huggingface_hub"], check=True)

import os, gc, glob, json, re, string, difflib, time
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import pandas as pd
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

## 1. Configuration

In [ ]:
CONFIG = {
    "languages": ["hi", "bn", "te", "en", "sw", "ar"],
    "models": [
        "Qwen/Qwen2.5-7B-Instruct",
        "google/gemma-2-9b-it",
        "meta-llama/Llama-3.1-8B-Instruct",
        "CohereLabs/aya-expanse-8b",
    ],
    "min_words": 4,              # queries shorter than this are skipped
    "max_new_tokens": 40,
    "batch_size": 16,
    "output_dir": "/kaggle/working/results" if os.path.exists("/kaggle") else "results",
}
LANG_NAMES = {"hi": "Hindi", "bn": "Bengali", "te": "Telugu", "en": "English", "sw": "Swahili", "ar": "Arabic"}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
if not torch.cuda.is_available():
    raise SystemExit("STOP: no GPU. Settings -> Accelerator -> GPU T4 x2, then run again.")
print("GPU:", torch.cuda.get_device_name(0))

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face")
except Exception as e:
    raise SystemExit(f"STOP: HF_TOKEN secret not attached (Add-ons -> Secrets -> tick HF_TOKEN). Details: {e}")

## 2. Load the sampled queries (same pairs files as notebook 03)

In [ ]:
PUNCT = string.punctuation + "।॥‘’“”«»،؛؟"


def norm(t):
    t = t.lower()
    t = "".join(" " if ch in PUNCT else ch for ch in t)
    return " ".join(t.split())


PROBES = []
for lang in CONFIG["languages"]:
    found = sorted(glob.glob(f"/kaggle/input/**/pairs_{lang}.jsonl", recursive=True)) + \
            sorted(glob.glob(f"**/pairs_{lang}.jsonl", recursive=True))
    if not found:
        print(f"WARNING: pairs_{lang}.jsonl not found")
        continue
    queries = {}
    with open(found[0], encoding="utf-8") as f:
        for line in f:
            p = json.loads(line)
            queries[p["qid"]] = p["query"]
    kept = 0
    for qid, q in sorted(queries.items()):
        words = q.split()
        if len(words) < CONFIG["min_words"]:
            continue
        cut = len(words) // 2
        PROBES.append({"lang": lang, "qid": qid, "query": q,
                       "prefix": " ".join(words[:cut]), "target": " ".join(words[cut:])})
        kept += 1
    print(f"{lang}: {kept}/{len(queries)} queries long enough to probe")

missing = [l for l in CONFIG["languages"] if l not in {p["lang"] for p in PROBES}]
if missing:
    raise SystemExit(f"STOP: pairs files missing for {missing}. Attach the missing input (notebook 2 v6 for hi/bn/te).")

## 3. Probe each model (greedy completion of the second half of each query)

In [ ]:
def build(tok, p):
    user = (f"The following is the beginning of a question from the MIRACL {LANG_NAMES[p['lang']]} "
            f"development set. Complete the question exactly as it appears in the dataset. "
            f"Output only the missing words.\n\n{p['prefix']}")
    msgs = [{"role": "user", "content": user}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


rows = []
out_csv = os.path.join(CONFIG["output_dir"], "contamination_probe.csv")
done_models = set()
if os.path.exists(out_csv):                                  # resume within a session
    prev = pd.read_csv(out_csv)
    rows = prev.to_dict("records")
    done_models = set(prev["model"])
for model_name in CONFIG["models"]:
    if model_name in done_models:
        print(f"{model_name}: already done, skipping")
        continue
    t0 = time.time()
    print(f"\n=== {model_name} ===")
    try:
        tok = AutoTokenizer.from_pretrained(model_name)
    except Exception as e:
        print(f"  FAILED to load ({e}) - skipping this model")
        continue
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                               bnb_4bit_compute_dtype=torch.float16))
    mdl.eval()
    bs = CONFIG["batch_size"]
    for i in range(0, len(PROBES), bs):
        batch = PROBES[i:i + bs]
        enc = tok([build(tok, p) for p in batch], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**enc, max_new_tokens=CONFIG["max_new_tokens"], do_sample=False,
                               pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for p, t in zip(batch, texts):
            comp = t.strip().split("\n")[0]
            nt, nc = norm(p["target"]), norm(comp)
            if nc.startswith(norm(p["prefix"])):          # model repeated the prefix first
                nc = nc[len(norm(p["prefix"])):].strip()
            rows.append({**p, "model": model_name, "completion": comp,
                         "exact": int(bool(nt) and nc.startswith(nt)),
                         "similarity": difflib.SequenceMatcher(None, nt, nc[:len(nt) + 5]).ratio()})
    print(f"  done in {time.time() - t0:.0f}s")
    pd.DataFrame(rows).to_csv(out_csv, index=False)          # save after every model
    del mdl, tok
    gc.collect()
    torch.cuda.empty_cache()

## 4. Results

In [ ]:
res = pd.DataFrame(rows)
res.to_csv(out_csv, index=False)
summary = res.groupby(["model", "lang"]).agg(queries=("exact", "size"),
                                             exact_rate=("exact", "mean"),
                                             mean_similarity=("similarity", "mean")).round(3)
summary.to_csv(os.path.join(CONFIG["output_dir"], "contamination_summary.csv"))
print(summary.to_string())
print("\nExamples of exact completions (if any):")
print(res[res.exact == 1][["model", "lang", "prefix", "target", "completion"]].head(15).to_string())